# IGF1R / calx-β — MM-GBSA binding free energy (post-MD)

Follow-up to [`md1_igf1r3qqu_calxbeta_gromacs.ipynb`](md1_igf1r3qqu_calxbeta_gromacs.ipynb), which runs independent replicates of the full calx-β–IGF1R kinase complex. Then continue with [`md3_igf1r3qqu_calxbeta_binding_inspection.ipynb`](md3_igf1r3qqu_calxbeta_binding_inspection.ipynb).

For every replicate in `data/md_calx_3qqu_full/rep{n}/`, this notebook estimates

$$\Delta G_{\mathrm{bind}} \approx \langle G_{\mathrm{complex}} \rangle - \langle G_{\mathrm{receptor}} \rangle - \langle G_{\mathrm{ligand}} \rangle$$

with **MM-GBSA** through **[gmx_MMPBSA](https://valdes-tresanco-ms.github.io/gmx_MMPBSA/)**. It implements Part B of `docs/md-review-response-spec.md`.

### Pipeline

1. **Frames:** md1's `rep{n}/md_pbc.xtc` — Protein only, receptor and calx-β clustered into one periodic image, chains checked whole. md2 does not re-process the trajectory.
2. **Contact check:** every selected frame must have receptor and calx-β heavy atoms within `MAX_INTERFACE_GAP_NM`. GB energies use no periodic images, so a frame with the partners in different box images would give ΔG ≈ 0 without any error. The notebook stops instead.
3. **Groups:** receptor = chains A and C, ligand = chain B (calx-β). Both come from the topology atom ranges.
4. **GB models (matched pairs):**
   - **GBn2** (`igb=8`, mbondi3 radii) is the main result.
   - **OBC-II** (`igb=5`, mbondi2 radii) is a sensitivity check.
   - The difference between the two measures part of the method uncertainty.
5. **Statistics:** the replicate is the unit of uncertainty. The notebook reports mean ΔG per replicate, then mean ± SD and SEM across replicates. Block averaging within each replicate is shown as a check. Neighbouring frames are correlated, so a per-frame SEM or bootstrap CI would understate the uncertainty, and neither is reported.

Outputs per replicate and model go to `rep{n}/mmpbsa/{model}/`. `rep{n}/mmpbsa/frame_times.csv` maps MM-GBSA frames to md1 times exactly, for md3.

### Caveats (reviewer comment #2)

- The **absolute** ΔG has systematic errors of tens of kcal/mol for an interface of this size. The causes are implicit solvent, the GB model and radii, the force field, and the configurational entropy that is not included. The absolute value is not an experimental binding free energy. Relative comparisons (ΔΔG, for example alanine scanning) cancel most of these errors.
- Results from GBn2 cannot be compared with the earlier pocket-run value (−39.8 kcal/mol). That run used OBC-II with mismatched radii, on a truncated receptor.

**Dependencies:** a dedicated conda env `gmxMMPBSA` (AmberTools + gmx_MMPBSA), installed on first use.

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def resolve_data_dir() -> Path:
    for candidate in (
        Path("../data"),
        Path("/workspace/data"),
        Path("data"),
    ):
        if (candidate / "md_calx_3qqu_full").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not find data/md_calx_3qqu_full — run md1 first.")


DATA_DIR = resolve_data_dir()
MD_ROOT = DATA_DIR / "md_calx_3qqu_full"
print(f"MD work dir: {MD_ROOT}")

In [ ]:
# --- Configuration ---
REPLICATES: list[int] | None = None  # None = every rep{n}/ that has md_pbc.xtc from md1
LIGAND_CHAIN = "B"  # calx-β; written last in md1's topology

RUN_MMPBSA = True  # False = prepare inputs and checks only
FORCE_RERUN = False  # True = recompute even when results for the same inputs exist

# GB model and radii must be a matched pair (reviewer comment #2). PBRadii: 3 = mbondi2, 4 = mbondi3.
GB_MODELS = {
    "gbn2": {"igb": 8, "pbradii": 4, "radii": "mbondi3", "label": "GBn2 (igb=8, mbondi3)"},
    "obc2": {"igb": 5, "pbradii": 3, "radii": "mbondi2", "label": "OBC-II (igb=5, mbondi2)"},
}
PRIMARY_MODEL = "gbn2"  # main result; md3 reads this one
RUN_MODELS = ["gbn2", "obc2"]  # obc2 is the sensitivity check
SALT_CONC_M = 0.150  # match MD ion concentration
FORCEFIELD = "amber99sb-ildn"

# Frames come from md1's md_pbc.xtc (stride set by md1's TRJCONV_DT_PS, 100 ps by default).
MMPBSA_SKIP_PS = 0.0  # ignore production frames before this time
MMPBSA_EVERY_N_FRAMES = 1  # use every n-th md_pbc frame (2 = half the frames, half the cost)
MAX_INTERFACE_GAP_NM = 0.5  # stop if receptor and calx-β heavy atoms are farther apart in any frame
N_BLOCKS = 5  # block averaging within a replicate (a check, not the reported uncertainty)

# None = all CPUs (capped at frame count). Set an int to pin the rank count.
MMPBSA_MPI_NPROCS: int | None = None
TMP_MMPBSA_ROOT = Path("/tmp/md_calx_3qqu_full_mmpbsa")  # scratch for gmx_MMPBSA runs

unknown = [m for m in RUN_MODELS if m not in GB_MODELS]
if unknown or PRIMARY_MODEL not in RUN_MODELS:
    raise ValueError(f"RUN_MODELS {RUN_MODELS} must use keys of GB_MODELS and include PRIMARY_MODEL")
print(
    f"models: {', '.join(GB_MODELS[m]['label'] for m in RUN_MODELS)}  (primary {PRIMARY_MODEL}); "
    f"salt {SALT_CONC_M} M; skip {MMPBSA_SKIP_PS:g} ps; every {MMPBSA_EVERY_N_FRAMES} frame(s)"
)

In [ ]:
def _itp_atom_count(itp_path: Path) -> int:
    """Count atoms in a GROMACS molecule .itp [ atoms ] section."""
    in_atoms = False
    count = 0
    for line in itp_path.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith("[ atoms ]"):
            in_atoms = True
            continue
        if in_atoms and stripped.startswith("["):
            break
        if in_atoms and re.match(r"\s*\d+\s+\S+", line):
            count += 1
    return count


def protein_chain_ranges(top_path: Path) -> dict[str, tuple[int, int]]:
    """0-based [start, end) atom ranges per protein chain, in topol.top order (calx-β last)."""
    section = None
    ranges: dict[str, tuple[int, int]] = {}
    start = 0
    for line in top_path.read_text().splitlines():
        stripped = line.split(";")[0].strip()
        if stripped.startswith("["):
            section = stripped.strip("[] ").lower()
            continue
        if section != "molecules" or not stripped:
            continue
        name, mult = stripped.split()[:2]
        if not name.startswith("Protein_chain_"):
            break
        n = _itp_atom_count(top_path.parent / f"topol_{name}.itp") * int(mult)
        ranges[name.rsplit("_", 1)[-1]] = (start, start + n)
        start += n
    if list(ranges)[-1:] != [LIGAND_CHAIN]:
        raise ValueError(f"calx-β (chain {LIGAND_CHAIN}) must be the last protein molecule in {top_path}")
    return ranges


@dataclass
class RepInputs:
    rep: int
    rep_dir: Path
    xtc: Path  # md1's md_pbc.xtc (Protein only, clustered)
    top_gro: Path  # md1's md_protein.gro (mdtraj topology for md_pbc.xtc)
    tpr: Path  # full-system production tpr
    top: Path
    ranges: dict[str, tuple[int, int]]
    mmpbsa_dir: Path


def discover_replicates() -> list[RepInputs]:
    found: list[RepInputs] = []
    for rep_dir in sorted(MD_ROOT.glob("rep*"), key=lambda p: int(p.name[3:]) if p.name[3:].isdigit() else 0):
        if not rep_dir.name[3:].isdigit():
            continue
        rep = int(rep_dir.name[3:])
        if REPLICATES is not None and rep not in REPLICATES:
            continue
        paths = [rep_dir / n for n in ("md_pbc.xtc", "md_protein.gro", "md.tpr", "topol.top")]
        if not all(p.exists() for p in paths):
            print(f"rep{rep}: skipped — run md1 processing first (needs md_pbc.xtc, md_protein.gro, md.tpr, topol.top)")
            continue
        found.append(
            RepInputs(rep, rep_dir, *paths, ranges=protein_chain_ranges(rep_dir / "topol.top"),
                      mmpbsa_dir=rep_dir / "mmpbsa")
        )
    if not found:
        raise FileNotFoundError(f"No replicate with md1 results under {MD_ROOT}")
    return found


def select_frames(inp: RepInputs) -> pd.DataFrame:
    """Frames of md_pbc.xtc used for MM-GBSA: mmpbsa_frame (1-based, as gmx_MMPBSA counts), traj_frame, time_ps.

    Every selected frame must have receptor and calx-β in contact; otherwise GB would see two
    separated proteins (ΔG ≈ 0) without any error.
    """
    traj = md.load(str(inp.xtc), top=str(inp.top_gro))
    keep = np.where(traj.time >= MMPBSA_SKIP_PS - 1e-6)[0][::MMPBSA_EVERY_N_FRAMES]
    if len(keep) == 0:
        raise ValueError(f"rep{inp.rep}: no frames after {MMPBSA_SKIP_PS} ps")
    if len(keep) > 1 and np.any(np.diff(keep) != MMPBSA_EVERY_N_FRAMES):
        raise ValueError("selected frames are not evenly spaced")

    heavy = np.array([a.element.symbol != "H" for a in traj.top.atoms])
    lig = np.arange(*inp.ranges[LIGAND_CHAIN])
    rec = np.concatenate([np.arange(*r) for c, r in inp.ranges.items() if c != LIGAND_CHAIN])
    sub = traj[keep]
    near = md.compute_neighbors(
        sub, MAX_INTERFACE_GAP_NM, query_indices=lig[heavy[lig]], haystack_indices=rec[heavy[rec]], periodic=False
    )
    apart = [int(i) for i, n in enumerate(near) if len(n) == 0]
    if apart:
        t = ", ".join(f"{sub.time[i]:g}" for i in apart[:10])
        raise RuntimeError(
            f"rep{inp.rep}: receptor and calx-β are more than {MAX_INTERFACE_GAP_NM} nm apart (no periodic images) "
            f"in {len(apart)} frame(s), e.g. t = {t} ps. Check md1's PBC processing before running MM-GBSA."
        )
    frames = pd.DataFrame(
        {"mmpbsa_frame": np.arange(1, len(keep) + 1), "traj_frame": keep + 1, "time_ps": traj.time[keep]}
    )
    inp.mmpbsa_dir.mkdir(parents=True, exist_ok=True)
    frames.to_csv(inp.mmpbsa_dir / "frame_times.csv", index=False)
    return frames


REPS = discover_replicates()
FRAMES: dict[int, pd.DataFrame] = {}
for inp in REPS:
    FRAMES[inp.rep] = select_frames(inp)
    f = FRAMES[inp.rep]
    sizes = {c: b - a for c, (a, b) in inp.ranges.items()}
    print(
        f"rep{inp.rep}: {len(f)} frames ({f['time_ps'].iloc[0]:.0f}–{f['time_ps'].iloc[-1]:.0f} ps), "
        f"chain atoms {sizes}; receptor–calx-β contact present in every frame"
    )

In [ ]:
def run_gmx(args: list[str], *, cwd: Path, stdin: str | None = None) -> str:
    """Run gmx in cwd; stdin is closed right away so prompts cannot wait on the notebook."""
    cmd = ["gmx", *args]
    print("$", " ".join(cmd), f"  (in {cwd.name})", flush=True)
    result = subprocess.run(cmd, cwd=cwd, input=stdin or "", text=True, capture_output=True, check=False)
    output = (result.stdout or "") + (result.stderr or "")
    if result.returncode != 0:
        raise RuntimeError(f"gmx failed ({result.returncode}): {output[-4000:]}")
    return output


def _write_ndx(path: Path, groups: dict[str, list[int]]) -> None:
    lines: list[str] = []
    for name, indices in groups.items():
        lines.append(f"[ {name} ]")
        for offset in range(0, len(indices), 15):
            lines.append(" ".join(f"{idx:5d}" for idx in indices[offset : offset + 15]))
        lines.append("")
    path.write_text("\n".join(lines).rstrip() + "\n")


def _write_protein_top(src: Path, dest: Path) -> None:
    """Copy topol.top but drop SOL/ions from [ molecules ] so it matches the Protein-only tpr."""
    out: list[str] = []
    in_molecules = False
    for line in src.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            in_molecules = stripped == "[ molecules ]"
            out.append(line)
            continue
        if in_molecules and stripped and not stripped.startswith(";"):
            if not stripped.split()[0].startswith("Protein_chain_"):
                continue
        out.append(line)
    dest.write_text("\n".join(out) + "\n")


def prepare_complex_inputs(inp: RepInputs) -> tuple[Path, Path]:
    """Protein-only tpr (matches md_pbc.xtc) and a Receptor/Ligand index (groups 0 and 1)."""
    complex_tpr = inp.mmpbsa_dir / "complex.tpr"
    if not complex_tpr.exists() or complex_tpr.stat().st_mtime < inp.tpr.stat().st_mtime:
        run_gmx(["make_ndx", "-f", "md.tpr", "-o", "mmpbsa/protein_index.ndx"], cwd=inp.rep_dir, stdin="q\n")
        run_gmx(
            ["convert-tpr", "-s", "md.tpr", "-n", "mmpbsa/protein_index.ndx", "-o", "mmpbsa/complex.tpr"],
            cwd=inp.rep_dir,
            stdin="Protein\n",
        )
    lig_lo, lig_hi = inp.ranges[LIGAND_CHAIN]
    index = inp.mmpbsa_dir / "mmpbsa.ndx"
    _write_ndx(index, {"Receptor": list(range(1, lig_lo + 1)), "Ligand": list(range(lig_lo + 1, lig_hi + 1))})
    return complex_tpr, index


COMPLEX_INPUTS = {inp.rep: prepare_complex_inputs(inp) for inp in REPS}
for inp in REPS:
    lig_lo, lig_hi = inp.ranges[LIGAND_CHAIN]
    print(f"rep{inp.rep}: complex.tpr + mmpbsa.ndx (Receptor atoms 1–{lig_lo}, Ligand atoms {lig_lo + 1}–{lig_hi})")

In [ ]:
def mmpbsa_input_text(model: str, frames: pd.DataFrame) -> str:
    """gmx_MMPBSA input: matched igb / PBRadii pair, frames as selected in frame_times.csv."""
    cfg = GB_MODELS[model]
    first, last = int(frames["traj_frame"].iloc[0]), int(frames["traj_frame"].iloc[-1])
    # Do not set gmx_path: gmx_MMPBSA uses it for *all* tools (cpptraj, tleap, gmx).
    return f"""&general
sys_name="calx_igf1r_{model}",
startframe={first}, endframe={last}, interval={MMPBSA_EVERY_N_FRAMES},
forcefields="{FORCEFIELD}",
PBRadii={cfg['pbradii']},
keep_files=0,
/
&gb
igb={cfg['igb']}, saltcon={SALT_CONC_M},
/
"""


for inp in REPS:
    for model in RUN_MODELS:
        out_dir = inp.mmpbsa_dir / model
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "mmpbsa.in").write_text(mmpbsa_input_text(model, FRAMES[inp.rep]))
print(f"Wrote mmpbsa.in for models {RUN_MODELS} in {len(REPS)} replicate(s)")

In [ ]:
# Dedicated gmxMMPBSA env (AmberTools + gmx_MMPBSA); keep separate from the Jupyter kernel.
MMPBSA_ENV = Path("/opt/conda/envs/gmxMMPBSA")
AUTO_INSTALL_MMPBSA = True


def _prepend_path(*dirs: Path) -> None:
    extra = [str(p) for p in dirs if p.is_dir()]
    if not extra:
        return
    current = os.environ.get("PATH", "")
    os.environ["PATH"] = os.pathsep.join([*extra, current])


def _activate_mmpbsa_env() -> None:
    """Put AmberTools + gmx on PATH and set AMBERHOME for this process."""
    os.environ["AMBERHOME"] = str(MMPBSA_ENV)
    _prepend_path(Path("/opt/conda/envs/gromacs/bin"), MMPBSA_ENV / "bin")


def _find_gmx_mmpbsa() -> str | None:
    _activate_mmpbsa_env()
    return shutil.which("gmx_MMPBSA")


def _run_install(cmd: list[str]) -> None:
    print("$", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, text=True, capture_output=True, check=False)
    if result.stdout:
        print(result.stdout[-3000:], flush=True)
    if result.returncode != 0:
        err = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"install failed ({result.returncode}):\n{err[-6000:]}")


def ensure_gmx_mmpbsa(*, auto_install: bool = AUTO_INSTALL_MMPBSA) -> str:
    """Return gmx_MMPBSA executable, installing a dedicated conda env if needed."""
    exe = _find_gmx_mmpbsa()
    if exe:
        return exe

    hint = (
        "  mamba create -n gmxMMPBSA python=3.11 -y\n"
        "  mamba install -n gmxMMPBSA -c conda-forge ambertools=24.8 gmx_mmpbsa mpi4py -y\n"
        "  # AmberTools 24.8 is the build used in this environment"
    )
    if not auto_install:
        raise FileNotFoundError(f"gmx_MMPBSA not found. Install with:\n{hint}")

    mamba = shutil.which("mamba") or "/opt/conda/bin/mamba"
    if not Path(mamba).exists():
        raise FileNotFoundError(f"mamba not found at {mamba}. Install AmberTools manually:\n{hint}")

    print(
        "gmx_MMPBSA missing — creating dedicated env gmxMMPBSA "
        "(Python 3.11 + AmberTools). This can take several minutes…",
        flush=True,
    )
    if not (MMPBSA_ENV / "bin" / "python").exists():
        _run_install([mamba, "create", "-n", "gmxMMPBSA", "python=3.11", "-y"])
    _run_install(
        [
            mamba,
            "install",
            "-n",
            "gmxMMPBSA",
            "-c",
            "conda-forge",
            "ambertools=24.8",
            "gmx_mmpbsa",
            "mpi4py",
            "-y",
        ]
    )

    exe = _find_gmx_mmpbsa()
    if not exe:
        raise FileNotFoundError(f"Install finished but gmx_MMPBSA is still not on PATH.\n{hint}")
    return exe


GMX_MMPBSA_EXE = ensure_gmx_mmpbsa()
cpptraj = shutil.which("cpptraj")
if not cpptraj:
    raise FileNotFoundError(
        f"gmx_MMPBSA is installed but cpptraj is not on PATH. "
        f"Expected {MMPBSA_ENV / 'bin' / 'cpptraj'}"
    )
def software_versions() -> dict[str, str]:
    """gmx_MMPBSA, AmberTools and ParmEd versions in the gmxMMPBSA env (for the Methods, spec A7)."""
    py = MMPBSA_ENV / "bin" / "python"
    probe = (
        "import importlib.metadata as m\n"
        "for d in ('gmx_MMPBSA', 'ParmEd', 'parmed', 'ambertools'):\n"
        "    try: print(d, m.version(d))\n"
        "    except Exception: pass\n"
    )
    out = subprocess.run([str(py), "-c", probe], capture_output=True, text=True, check=False).stdout
    versions = dict(line.split(" ", 1) for line in out.splitlines() if " " in line)
    conda_list = subprocess.run(
        [shutil.which("mamba") or "/opt/conda/bin/mamba", "list", "-n", "gmxMMPBSA", "^ambertools$"],
        capture_output=True, text=True, check=False,
    ).stdout
    m = re.search(r"^ambertools\s+(\S+)", conda_list, flags=re.M)
    if m:
        versions["ambertools"] = m.group(1)
    gmx = subprocess.run(["gmx", "--version"], capture_output=True, text=True, check=False).stdout
    m = re.search(r"GROMACS version:\s*(\S+)", gmx)
    if m:
        versions["gromacs"] = m.group(1)
    return versions


SOFTWARE = software_versions()
print(f"Using: {GMX_MMPBSA_EXE}")
print(f"cpptraj: {cpptraj}")
print(f"AMBERHOME: {os.environ.get('AMBERHOME')}")
print("versions:", SOFTWARE)

In [ ]:
def _mpi_nprocs(n_frames: int) -> int:
    """Ranks = available CPUs, but never more ranks than MMPBSA frames."""
    requested = MMPBSA_MPI_NPROCS if MMPBSA_MPI_NPROCS is not None else (os.cpu_count() or 1)
    return max(1, min(int(requested), max(1, int(n_frames))))


def _mpirun_prefix(nprocs: int) -> list[str]:
    """Launch gmx_MMPBSA with the MPI in the gmxMMPBSA env (not GROMACS Open MPI).

    Open MPI refuses to run as root without --allow-run-as-root, which `mpirun --help` does not
    always list, so Open MPI is detected from `--version` or a sibling prterun binary.
    """
    candidates = [MMPBSA_ENV / "bin" / "mpirun", MMPBSA_ENV / "bin" / "mpiexec", Path(shutil.which("mpirun") or "")]
    mpirun = next((p for p in candidates if p and p.is_file()), None)
    if mpirun is None:
        raise FileNotFoundError(f"mpirun not found in {MMPBSA_ENV / 'bin'}. Install mpi4py in the gmxMMPBSA env.")
    prefix = [str(mpirun), "-np", str(nprocs)]
    probe = subprocess.run([str(mpirun), "--version"], capture_output=True, text=True, check=False)
    ident = ((probe.stdout or "") + (probe.stderr or "")).lower()
    openmpi = any(s in ident for s in ("open mpi", "openmpi", "prte", "prterun")) or (mpirun.parent / "prterun").is_file()
    if os.geteuid() == 0 and openmpi:
        prefix.append("--allow-run-as-root")
    return prefix


def _results_are_current(out_dir: Path, inp: RepInputs) -> bool:
    """Reuse results when they are newer than md_pbc.xtc and were made from the same mmpbsa.in."""
    csv, used = out_dir / "FINAL_RESULTS_MMPBSA.csv", out_dir / "mmpbsa.in.used"
    if FORCE_RERUN or not csv.exists() or not used.exists():
        return False
    return csv.stat().st_mtime >= inp.xtc.stat().st_mtime and used.read_text() == (out_dir / "mmpbsa.in").read_text()


def run_gmx_mmpbsa(inp: RepInputs, model: str) -> Path:
    """gmx_MMPBSA for one replicate and GB model; results in rep{n}/mmpbsa/{model}/."""
    out_dir = inp.mmpbsa_dir / model
    if _results_are_current(out_dir, inp):
        print(f"rep{inp.rep} {model}: reusing results (same inputs; FORCE_RERUN=True to recompute)")
        return out_dir
    complex_tpr, index = COMPLEX_INPUTS[inp.rep]
    work = TMP_MMPBSA_ROOT / f"rep{inp.rep}" / model
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True)
    for src in (complex_tpr, index, inp.xtc, out_dir / "mmpbsa.in"):
        shutil.copy2(src, work / src.name)
    _write_protein_top(inp.top, work / "topol.top")
    for itp in inp.rep_dir.glob("*.itp"):
        shutil.copy2(itp, work / itp.name)

    n_frames = len(FRAMES[inp.rep])
    args = [
        "-O", "-nogui",  # no PyQt in the container
        "-i", "mmpbsa.in", "-cs", complex_tpr.name, "-cp", "topol.top", "-ci", index.name,
        "-cg", "0", "1", "-ct", inp.xtc.name,
        "-o", "FINAL_RESULTS_MMPBSA.dat", "-eo", "FINAL_RESULTS_MMPBSA.csv",
    ]
    nprocs = _mpi_nprocs(n_frames)
    cmd = [GMX_MMPBSA_EXE, *args] if nprocs <= 1 else [*_mpirun_prefix(nprocs), GMX_MMPBSA_EXE, "MPI", *args]
    env = os.environ.copy()
    env.update({"AMBERHOME": str(MMPBSA_ENV), "TMPDIR": str(work)})
    env["PATH"] = os.pathsep.join([str(MMPBSA_ENV / "bin"), "/opt/conda/envs/gromacs/bin", env.get("PATH", "")])
    print(f"rep{inp.rep} {model}: gmx_MMPBSA on {n_frames} frames, {nprocs} MPI rank(s)", flush=True)
    log_path = work / "gmx_MMPBSA_run.log"
    with log_path.open("w", encoding="utf-8") as log_fh:
        proc = subprocess.run(cmd, cwd=work, env=env, stdout=log_fh, stderr=subprocess.STDOUT, text=True)
    log = log_path.read_text(encoding="utf-8", errors="replace")
    results = work / "FINAL_RESULTS_MMPBSA.csv"
    gui_only_fail = "Unable to start gmx_MMPBSA_ana" in log and results.exists()
    if proc.returncode != 0 and not gui_only_fail:
        raise RuntimeError(f"rep{inp.rep} {model}: gmx_MMPBSA failed ({proc.returncode}):\n{log[-6000:]}")
    for name in ("FINAL_RESULTS_MMPBSA.dat", "FINAL_RESULTS_MMPBSA.csv", "gmx_MMPBSA.log", "gmx_MMPBSA_run.log"):
        if (work / name).exists():
            shutil.copy2(work / name, out_dir / name)
    shutil.copy2(out_dir / "mmpbsa.in", out_dir / "mmpbsa.in.used")
    return out_dir


if RUN_MMPBSA:
    for inp in REPS:
        for model in RUN_MODELS:
            run_gmx_mmpbsa(inp, model)
    print("gmx_MMPBSA finished.")
else:
    print("RUN_MMPBSA=False — inputs prepared; set True to execute.")

In [ ]:
def delta_terms(csv_path: Path) -> pd.DataFrame:
    """Per-frame ΔG terms from the 'Delta Energy Terms' block of FINAL_RESULTS_MMPBSA.csv."""
    lines = csv_path.read_text().splitlines()
    start = next((i for i, ln in enumerate(lines) if ln.strip() == "Delta Energy Terms"), None)
    if start is None:
        raise ValueError(f"No 'Delta Energy Terms' block in {csv_path}")
    block: list[str] = []
    for ln in lines[start + 1 :]:
        if not ln.strip() or (ln.endswith("Energy Terms") and not ln.startswith("Frame")):
            break
        block.append(ln)
    df = pd.read_csv(StringIO("\n".join(block)))
    if "TOTAL" not in df.columns:
        raise ValueError(f"Delta Energy Terms in {csv_path} has no TOTAL column")
    return df


def settings_that_ran(out_dir: Path) -> dict[str, str]:
    """GB settings as echoed by gmx_MMPBSA (not as written in mmpbsa.in)."""
    text = ""
    for name in ("FINAL_RESULTS_MMPBSA.dat", "gmx_MMPBSA.log"):
        if (out_dir / name).exists():
            text += (out_dir / name).read_text(errors="replace") + "\n"
    found: dict[str, str] = {}
    for key in ("igb", "PBRadii", "saltcon"):
        m = re.search(rf"\b{key}\s*=\s*([0-9.]+)", text)
        if m:
            found[key] = m.group(1)
    m = re.search(r"(mbondi\d?|bondi)\b", text)
    if m:
        found["radii_reported"] = m.group(1)
    return found


def block_sem(values: np.ndarray, n_blocks: int = N_BLOCKS) -> float:
    """SEM from block means — accounts for correlated neighbouring frames better than per-frame SEM."""
    if len(values) < 2 * n_blocks:
        return float("nan")
    means = [b.mean() for b in np.array_split(values, n_blocks)]
    return float(np.std(means, ddof=1) / np.sqrt(n_blocks))


def summarize_run(inp: RepInputs, model: str) -> tuple[dict, pd.DataFrame]:
    out_dir = inp.mmpbsa_dir / model
    terms = delta_terms(out_dir / "FINAL_RESULTS_MMPBSA.csv")
    frames = FRAMES[inp.rep]
    if len(terms) != len(frames):
        raise ValueError(f"rep{inp.rep} {model}: {len(terms)} MM-GBSA frames but {len(frames)} selected frames")
    per_frame = frames.copy()
    per_frame["delta_g_kcal"] = terms["TOTAL"].to_numpy(dtype=float)
    per_frame.to_csv(out_dir / "delta_g_per_frame.csv", index=False)

    ran = settings_that_ran(out_dir)
    cfg = GB_MODELS[model]
    if ran.get("igb") and int(float(ran["igb"])) != cfg["igb"]:
        raise RuntimeError(f"rep{inp.rep} {model}: gmx_MMPBSA ran igb={ran['igb']}, expected {cfg['igb']}")
    if ran.get("PBRadii") and int(float(ran["PBRadii"])) != cfg["pbradii"]:
        raise RuntimeError(f"rep{inp.rep} {model}: gmx_MMPBSA ran PBRadii={ran['PBRadii']}, expected {cfg['pbradii']}")

    v = per_frame["delta_g_kcal"].to_numpy()
    half = len(v) // 2
    row = {
        "replicate": inp.rep,
        "model": model,
        "frames": len(v),
        "time_ps_start": float(frames["time_ps"].iloc[0]),
        "time_ps_end": float(frames["time_ps"].iloc[-1]),
        "delta_g_mean": float(v.mean()),
        "delta_g_sd_frames": float(v.std(ddof=1)) if len(v) > 1 else float("nan"),
        "delta_g_block_sem": block_sem(v),
        "late_minus_early": float(v[half:].mean() - v[:half].mean()) if len(v) > 1 else float("nan"),
        "frames_dg_ge_0": int((v >= 0).sum()),
    }
    manifest = {
        **row,
        "gb_model": cfg["label"],
        "settings_requested": {"igb": cfg["igb"], "PBRadii": cfg["pbradii"], "radii": cfg["radii"], "saltcon": SALT_CONC_M},
        "settings_reported_by_gmx_mmpbsa": ran,
        "software": SOFTWARE,
        "source_trajectory": str(inp.xtc),
        "saved_at": datetime.now(timezone.utc).isoformat(),
    }
    (out_dir / "mmpbsa_results_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    return row, per_frame


rows: list[dict] = []
PER_FRAME: dict[tuple[int, str], pd.DataFrame] = {}
for inp in REPS:
    for model in RUN_MODELS:
        if not (inp.mmpbsa_dir / model / "FINAL_RESULTS_MMPBSA.csv").exists():
            print(f"rep{inp.rep} {model}: no results yet")
            continue
        row, per_frame = summarize_run(inp, model)
        rows.append(row)
        PER_FRAME[(inp.rep, model)] = per_frame
        print(f"rep{inp.rep} {model}: settings reported by gmx_MMPBSA {settings_that_ran(inp.mmpbsa_dir / model)}")

RESULTS = pd.DataFrame(rows)
if RESULTS.empty:
    print("No MM-GBSA results yet — run with RUN_MMPBSA=True.")
else:
    display(Markdown("**Per replicate** (ΔG in kcal/mol; SD over frames; block SEM is a within-replicate check):"))
    display(RESULTS.round(2))

    across = (
        RESULTS.groupby("model")["delta_g_mean"]
        .agg(n="count", mean="mean", sd=lambda s: s.std(ddof=1))
        .assign(sem=lambda d: d["sd"] / np.sqrt(d["n"]))
    )
    display(Markdown("**Across replicates** (the replicate is the unit of uncertainty; SEM = SD/√n):"))
    display(across.round(2))

    if {PRIMARY_MODEL} < set(RESULTS["model"]) and len(set(RESULTS["model"])) > 1:
        wide = RESULTS.pivot(index="replicate", columns="model", values="delta_g_mean")
        other = [m for m in RUN_MODELS if m != PRIMARY_MODEL][0]
        wide[f"{other} − {PRIMARY_MODEL}"] = wide[other] - wide[PRIMARY_MODEL]
        display(Markdown("**GB-model sensitivity** (difference per replicate = part of the method uncertainty):"))
        display(wide.round(2))

    fig, ax = plt.subplots(figsize=(9, 3.2))
    for (rep, model), pf in sorted(PER_FRAME.items()):
        if model == PRIMARY_MODEL:
            ax.plot(pf["time_ps"] / 1000.0, pf["delta_g_kcal"], lw=0.8, label=f"rep{rep}")
    ax.axhline(0, color="0.4", ls=":", lw=1)
    ax.set_xlabel("Production time (ns)")
    ax.set_ylabel("ΔG (kcal/mol)")
    ax.set_title(f"MM-GBSA per frame — {GB_MODELS[PRIMARY_MODEL]['label']}")
    ax.legend(fontsize=7, ncol=5)
    plt.tight_layout()
    plt.show()
    plt.close("all")

    summary = {
        "models": {m: GB_MODELS[m] for m in RUN_MODELS},
        "primary_model": PRIMARY_MODEL,
        "per_replicate": RESULTS.to_dict(orient="records"),
        "across_replicates": across.reset_index().to_dict(orient="records"),
        "software": SOFTWARE,
        "saved_at": datetime.now(timezone.utc).isoformat(),
    }
    (MD_ROOT / "mmpbsa_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(f"Saved {MD_ROOT / 'mmpbsa_summary.json'}")

## Interpretation

The next cell builds this section from the per-replicate results: mean ΔG per replicate, the spread across replicates, the GB-model difference, and the frames with ΔG ≥ 0.

In [ ]:
def write_mmpbsa_interpretation(results: pd.DataFrame) -> str:
    """Interpretation built on replicates, with the reviewer #2 caveats on the absolute ΔG."""
    if results.empty:
        text = "## Interpretation\n\nNo MM-GBSA results yet. Set `RUN_MMPBSA = True` and re-run."
        display(Markdown(text))
        return text

    prim = results[results["model"] == PRIMARY_MODEL].sort_values("replicate")
    n = len(prim)
    mean = float(prim["delta_g_mean"].mean())
    sd = float(prim["delta_g_mean"].std(ddof=1)) if n > 1 else float("nan")
    sem = sd / np.sqrt(n) if n > 1 else float("nan")
    all_neg = bool((prim["delta_g_mean"] < 0).all())
    length_ns = float((prim["time_ps_end"] - prim["time_ps_start"]).min()) / 1000.0
    label = GB_MODELS[PRIMARY_MODEL]["label"]

    per_rep = "; ".join(f"rep{int(r.replicate)} {r.delta_g_mean:.1f}" for r in prim.itertuples())
    if n > 1:
        headline = (
            f"Across **{n} replicates** ({label}, no entropy), ΔG = **{mean:.1f} ± {sem:.1f} kcal/mol** "
            f"(mean ± SEM across replicates; SD {sd:.1f}). Per replicate: {per_rep}."
        )
    else:
        headline = (
            f"Only **one replicate** so far ({label}, no entropy): mean ΔG = **{mean:.1f} kcal/mol**. "
            "An uncertainty needs at least 3 replicates."
        )

    sensitivity = ""
    others = [m for m in RUN_MODELS if m != PRIMARY_MODEL]
    if others:
        other = results[results["model"] == others[0]].set_index("replicate")["delta_g_mean"]
        diff = (other - prim.set_index("replicate")["delta_g_mean"]).dropna()
        if len(diff):
            sensitivity = (
                f"- **GB model:** {GB_MODELS[others[0]]['label']} differs from {label} by "
                f"{diff.mean():+.1f} kcal/mol on average (range {diff.min():+.1f} to {diff.max():+.1f}). "
                "This difference is part of the method uncertainty and should be reported with the result.\n"
            )

    flips = int(prim["frames_dg_ge_0"].sum())
    frames = int(prim["frames"].sum())
    sign = (
        "The mean is negative in every replicate."
        if all_neg
        else "The mean is **not** negative in every replicate — inspect those replicates in md3 before drawing conclusions."
    )
    short = (
        f"\n- **Sampling:** the shortest replicate covers {length_ns:.1f} ns; prefer ≥ 50–100 ns per replicate."
        if length_ns < 50
        else ""
    )

    flip_note = (
        f"No frame has ΔG ≥ 0 ({frames} frames)."
        if flips == 0
        else f"{flips} of {frames} frames have ΔG ≥ 0; md3 checks whether those frames are energy fluctuations "
        "or loss of the docked geometry."
    )
    text = f"""## Interpretation

{headline} {sign} {flip_note}

### How to read this number

- **Absolute value:** MM-GBSA here leaves out configurational entropy and uses an implicit solvent model. For an interface of this size, the systematic error of the absolute ΔG is tens of kcal/mol — about the size of the value itself. The number is **not** an experimental binding free energy and must not be converted to a Kd.
- **Uncertainty:** the spread across independent replicates is the reported uncertainty. A per-frame SEM or bootstrap CI over correlated frames would be much smaller and does not include the systematic error above, so it is not reported.
{sensitivity}- **Not comparable with earlier numbers:** the pocket-run value (−39.8 kcal/mol) used OBC-II with mismatched radii on a truncated receptor.{short}

### What would make the computational result stronger

- **Relative test (ΔΔG):** alanine scanning of the residues whose mutation blocked binding experimentally, with control residues away from the interface. Most systematic errors cancel in ΔΔG (spec B2 item 2).
- **Entropy:** normal-mode analysis on a subset of frames, reported as a separate line (spec B2 item 3).
- **Solvent model:** an MM-PBSA cross-check on a subset of frames (spec B2 item 4).
"""
    display(Markdown(text))
    (MD_ROOT / "mmpbsa_interpretation.md").write_text(text)
    print(f"Wrote {MD_ROOT / 'mmpbsa_interpretation.md'}")
    return text


_ = write_mmpbsa_interpretation(RESULTS)

In [ ]:
! jupyter nbconvert --to html --no-input --embed-images md2_igf1r3qqu_calxbeta_pbsa_gbsa.ipynb